# Swaram: studio voice (train the big model on one voice)

Started by the Swaram app. It trains IndicF5 itself on your recordings (about
2.5 hours on Kaggle's free GPU). The result stays in this notebook's output and
is used when the app renders scripts on Kaggle (Studio render).

In [ ]:
# 1. Setup (about 5 minutes)
import os, subprocess, glob, json, base64, zipfile
if not os.environ.get("HF_TOKEN"):
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
WORK = "/tmp/render"
OUT = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(f"set -o pipefail; {cmd}", shell=True, check=True, executable="/bin/bash")

sh(f"git clone -q --depth 1 https://github.com/Dattaharshithreddy/Swaram.git {WORK}/Swaram || true")
sh(f"git clone -q --depth 1 https://github.com/AI4Bharat/IndicF5.git {WORK}/IndicF5 || true")
sh("pip install -q x-transformers torchdiffeq vocos==0.1.0 einops jieba pypinyin ema-pytorch librosa soundfile safetensors onnx onnxruntime")
os.environ["PYTHONPATH"] = f"{WORK}/IndicF5:{WORK}/Swaram/edge"
import torch; print("GPU:", torch.cuda.is_available())


In [ ]:
# 2. Your recordings (the private dataset the app uploaded)
assert "APP_PRO" in globals(), "Start this from the Swaram app (Voices -> Studio voice)."
DATA = f"{WORK}/data"
os.makedirs(f"{DATA}/wavs", exist_ok=True)
# Kaggle unpacks uploaded zips, so look for the words file (or a zip that was kept).
metas = glob.glob("/kaggle/input/**/metadata.csv", recursive=True)
packs = glob.glob("/kaggle/input/**/swaram-pro-*.zip", recursive=True)
if metas:
    PACK = os.path.dirname(metas[0])
elif packs:
    with zipfile.ZipFile(packs[0]) as z:
        z.extractall(f"{WORK}/pack")
    PACK = f"{WORK}/pack"
else:
    print(glob.glob("/kaggle/input/**", recursive=True)[:50])
    raise AssertionError("The recordings dataset is not attached.")
import librosa, soundfile as sf
rows = []
for line in open(f"{PACK}/metadata.csv", encoding="utf-8").read().splitlines():
    name, text = line.split("|", 1)
    src = f"{PACK}/clips/{name}"
    if not os.path.exists(src):
        found = glob.glob(f"{PACK}/**/{name}", recursive=True)
        if not found:
            continue
        src = found[0]
    wav, _ = librosa.load(src, sr=24000, mono=True)
    out = name.rsplit(".", 1)[0] + ".wav"
    sf.write(f"{DATA}/wavs/{out}", wav, 24000)
    rows.append(f"{out}|{text}")
open(f"{DATA}/metadata.csv", "w", encoding="utf-8").write("\n".join(rows))
print(len(rows), "recordings for", APP_PRO.get("name"))


In [ ]:
# 3. Train (about APP_PRO["hours"] hours)
sh(f"cd {WORK}/Swaram/cloud && python finetune.py --data {DATA} --out {OUT} --id {APP_PRO['id']} --hours {APP_PRO.get('hours', 2.5)}")
json.dump({"id": APP_PRO["id"], "name": APP_PRO.get("name", ""), "file": f"voice-model-{APP_PRO['id']}.safetensors"}, open(f"{OUT}/studio-voice.json", "w"))
from IPython.display import Audio, display
display(Audio(f"{OUT}/sample.wav"))
